# 05 — Integrador y verificación
El integrador online requiere modelo, kit, índice y LangSmith. Esta demostración local verifica rechazo de una respuesta inconsistente.

In [ ]:
from compraslab.schemas import SolicitudCompra, RespuestaCompra
from compraslab.verification import verificar
r=RespuestaCompra(estado="recomendacion", mensaje="Oferta sin respaldo",dataset_id="demo", solicitud=SolicitudCompra(producto_id="PROD-1",cantidad=2,presupuesto_centavos=10000,plazo_max_dias=3), proveedor_id="A",total_centavos=5000)
print(verificar(r).model_dump())

La prueba `tests/test_integrator.py` ejecuta dos propuestas con un executor guionado: la propuesta rechazada no entra al historial. La integración con modelo, NLI remoto y LangSmith requiere credenciales y evidencias reales.

In [ ]:
from tempfile import TemporaryDirectory
from pathlib import Path
from compraslab.data import crear_catalogo, ofertas_producto
from compraslab.business import comparar_ofertas
from compraslab.schemas import SolicitudCompra, RespuestaCompra, Evidencia
from compraslab.verification import verificar
with TemporaryDirectory() as d:
    cat=crear_catalogo(Path(d), cantidad=3)
    solicitud=SolicitudCompra(producto_id="PROD-0001",cantidad=1,presupuesto_centavos=10000000,plazo_max_dias=30)
    alternativas=comparar_ofertas(solicitud,ofertas_producto(cat,"PROD-0001"))
    ganador=alternativas[0]
    resp=RespuestaCompra(estado="recomendacion",mensaje="demo",dataset_id=cat.dataset_id,solicitud=solicitud,producto_id="PROD-0001",proveedor_id=ganador.oferta.proveedor_id,total_centavos=ganador.total_centavos,alternativas=alternativas,evidencias=[Evidencia(tipo="catalogo",fuente_id=a.oferta.id,dataset_id=cat.dataset_id,texto="oferta") for a in alternativas])
    print(verificar(resp,cat).model_dump())